In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/01-Research/TRACE"
DIRS = [
    "data/public", "data/generated", "data/packaged",
    "models/lora", "models/onnx", "outputs/logs", "outputs/benchmarks",
    "api", "dashboard"
]
for d in DIRS:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)

print(f"✅ TRACE workspace ready at: {BASE_DIR}")

Mounted at /content/drive
✅ TRACE workspace ready at: /content/drive/MyDrive/01-Research/TRACE


In [2]:
!pip install -q biopython dnachisel datasets transformers peft accelerate scikit-learn networkx shap captum fastapi uvicorn streamlit wandb matplotlib seaborn
print("✅ Dependencies installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.2/147.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 104.8 MB/s eta 0:00:00
✅ Dependencies installed


#  Cell 3 : Download Public Data

In [12]:
import requests
import json
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

public_dir = Path(BASE_DIR) / "data/public"
public_dir.mkdir(parents=True, exist_ok=True)

# Exponential backoff mitigates Colab network drops & server-side throttling
session = requests.Session()
retry = Retry(total=5, backoff_factor=1.5, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retry))

url = "https://rest.uniprot.org/uniprotkb/stream"

print("📥 Downloading UniProt Tox-Prot (reviewed toxins)...")
tox_params = {"query": "reviewed:true AND keyword:Toxin", "format": "fasta", "compressed": "false", "size": 600}
with session.get(url, params=tox_params, stream=True, timeout=180) as r:
    r.raise_for_status()
    with open(public_dir / "toxprot.fasta", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("📥 Downloading hard negatives (human/E. coli reviewed proteins)...")
# Simplified query avoids 400 errors from boolean negation; toxins are negligible in these proteomes
neg_params = {"query": "reviewed:true AND (organism_id:9606 OR organism_id:83333)", "format": "fasta", "compressed": "false", "size": 600}
with session.get(url, params=neg_params, stream=True, timeout=180) as r:
    r.raise_for_status()
    with open(public_dir / "hard_negatives.fasta", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("📥 Saving PROSITE/InterPro toxic motifs (Stage 1 prefilter)...")
motifs = {
    "HExxH_metalloprotease": r"H.E..H",
    "Ricin_lectin_domain": r"[QK]X[WY]X[ST]X[DE]",
    "BoNT_SNAP25_cleavage": r"[RK]..[QE]..[LIV]",
    "Anthrax_LF_zinc_bind": r"H.E.X.H"
}
(public_dir / "motif_patterns.json").write_text(json.dumps(motifs, indent=2))

print(f"✅ Public data saved to {public_dir}")

📥 Downloading UniProt Tox-Prot (reviewed toxins)...
📥 Downloading hard negatives (human/E. coli reviewed proteins)...
📥 Saving PROSITE/InterPro toxic motifs (Stage 1 prefilter)...
✅ Public data saved to /content/drive/MyDrive/01-Research/TRACE/data/public


# Cell 4 : Generate Adversarial & Fragmented Data

In [13]:
import random
import json
import logging
import dnachisel as dc
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import PairwiseAligner
from Bio.Data import CodonTable
from pathlib import Path

gen_dir = Path(BASE_DIR) / "data/generated"
gen_dir.mkdir(parents=True, exist_ok=True)

# fasta-pearson ignores UniProt header comments
toxins = list(SeqIO.parse(public_dir / "toxprot.fasta", "fasta-pearson"))[:20]
benign = list(SeqIO.parse(public_dir / "igem_benign.fasta", "fasta-pearson"))
toxin_ids = {rec.id for rec in toxins}  # O(1) lookup; avoids SeqRecord __eq__

# Suppress DNAChisel tqdm/log noise during batch optimization
logging.getLogger("dnachisel").setLevel(logging.ERROR)

CODON_MAP = {aa: codons[0] for aa, codons in CodonTable.unambiguous_dna_by_name["Standard"].forward_table.items()}
CODON_MAP['*'] = 'TAA'

def back_translate(protein_seq):
    return "".join(CODON_MAP.get(aa, 'GCT') for aa in protein_seq)

def fragment_dna(seq, frag_len=(50, 100), overlap=20):
    step = random.randint(frag_len[0], frag_len[1]) - overlap
    return [seq[i:min(i + random.randint(frag_len[0], frag_len[1]), len(seq))]
            for i in range(0, len(seq) - overlap, step)]

def codon_optimize(seq, host="e_coli"):
    problem = dc.DnaOptimizationProblem(
        sequence=seq,
        constraints=[dc.EnforceTranslation()],
        objectives=[dc.CodonOptimize(species=host)]
    )
    problem.resolve_constraints()
    problem.optimize()
    return str(problem.sequence)

def simulate_gibson_overlaps(frags):
    return [f + frags[i+1][:20] if i < len(frags)-1 else f for i, f in enumerate(frags)]

print("🧬 Generating fragmented oligo carts (Stage 2 input)...")
carts = [
    {
        "order_id": f"ORD_{rec.id}",
        "customer_hash": f"hash_{abs(hash(rec.id)) % 1000}",
        "fragments": simulate_gibson_overlaps(fragment_dna(codon_optimize(back_translate(str(rec.seq))))),
        "assembly_method": "gibson",
        "ground_truth": "toxin" if rec.id in toxin_ids else "benign"  # Fixed: explicit ID comparison
    }
    for rec in toxins[:5] + benign[:5]
]
(gen_dir / "test_carts.json").write_text(json.dumps(carts, indent=2))

print("🔄 Generating AI-paraphrased variants (Stage 3 adversarial set)...")
from transformers import pipeline
import torch

pipe = pipeline("fill-mask", model="facebook/esm2_t12_35M_UR50D", device=0 if torch.cuda.is_available() else -1)
aligner = PairwiseAligner()
aligner.mode = "global"

paraphrases = []
for rec in toxins[:5]:
    wt = str(rec.seq)
    for _ in range(3):
        seq_list = list(wt)
        for idx in random.sample(range(len(seq_list)), int(len(seq_list) * 0.4)):
            masked = "".join(seq_list[:idx]) + "<mask>" + "".join(seq_list[idx+1:])
            seq_list[idx] = pipe(masked, top_k=1)[0]["token_str"]
        var = "".join(seq_list)
        if aligner.align(wt, var)[0].score / len(wt) <= 0.3:
            paraphrases.append(SeqRecord(Seq(var), id=f"{rec.id}_AIv", description="ai_paraphrase"))
SeqIO.write(paraphrases, gen_dir / "ai_paraphrases.fasta", "fasta")

print(f"✅ Generated data saved to {gen_dir}")

🧬 Generating fragmented oligo carts (Stage 2 input)...


location:   0%|          | 0/549 [00:00<?, ?it/s, now=0-3] 


🔄 Generating AI-paraphrased variants (Stage 3 adversarial set)...


Loading weights:   0%|          | 0/214 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Generated data saved to /content/drive/MyDrive/01-Research/TRACE/data/generated


# Cell 5: Robust LoRA Dataset Packaging

In [14]:
import random
import pandas as pd
from Bio import SeqIO
from datasets import Dataset, DatasetDict
from pathlib import Path

pkg_dir = Path(BASE_DIR) / "data/packaged"
pkg_dir.mkdir(parents=True, exist_ok=True)

print("📦 Packaging LoRA training dataset (family-held-out, min 3 seqs/family)...")
toxins = list(SeqIO.parse(public_dir / "toxprot.fasta", "fasta-pearson"))
negatives = list(SeqIO.parse(public_dir / "hard_negatives.fasta", "fasta-pearson"))
paraphrases = list(SeqIO.parse(Path(BASE_DIR) / "data/generated/ai_paraphrases.fasta", "fasta-pearson"))

# Extract family prefix (first 6 chars of UniProt ID) for leakage-controlled splits
def get_family(rec): return rec.id.split("_")[0][:6]

rows = [
    {"sequence": str(rec.seq), "label": 1, "family": get_family(rec)} for rec in toxins + paraphrases
] + [
    {"sequence": str(rec.seq), "label": 0, "family": f"NEG_{i//50}"} for i, rec in enumerate(negatives)
]

df = pd.DataFrame(rows)

# Enforce minimum family representation to prevent split collapse
family_counts = df["family"].value_counts()
valid_families = family_counts[family_counts >= 3].index
df = df[df["family"].isin(valid_families)]

# Family-held-out split (80/20)
families = df["family"].unique()
random.shuffle(families)
split_idx = int(len(families) * 0.8)
train_fams, val_fams = families[:split_idx], families[split_idx:]

train_ds = Dataset.from_pandas(df[df["family"].isin(train_fams)][["sequence", "label"]])
val_ds = Dataset.from_pandas(df[df["family"].isin(val_fams)][["sequence", "label"]])

DatasetDict({"train": train_ds, "validation": val_ds}).save_to_disk(str(pkg_dir))
print(f"✅ Packaged dataset saved to {pkg_dir} | Train: {len(train_ds)} | Val: {len(val_ds)}")

📦 Packaging LoRA training dataset (family-held-out, min 3 seqs/family)...


Saving the dataset (0/1 shards):   0%|          | 0/26246 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6280 [00:00<?, ? examples/s]

✅ Packaged dataset saved to /content/drive/MyDrive/01-Research/TRACE/data/packaged | Train: 26246 | Val: 6280


# Cell 6: LoRA Fine-Tuning (ESM-2 650M + PEFT)

In [20]:
import torch
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
    Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score
from datasets import load_from_disk

# Load packaged dataset & tokenizer
ds = load_from_disk(str(pkg_dir))
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")

# Vectorized tokenization; truncation conserves T4 VRAM
ds = ds.map(lambda b: tokenizer(b["sequence"], truncation=True, padding="max_length", max_length=512), batched=True)
ds = ds.rename_column("label", "labels")

# Compute class weights to handle toxin/benign imbalance
flat_labels = np.array(ds["train"]["labels"])
class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=flat_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32).cuda()

# Load base model & inject LoRA adapters
model = AutoModelForSequenceClassification.from_pretrained(
    "facebook/esm2_t33_650M_UR50D", num_labels=2, ignore_mismatched_sizes=True
)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "key", "value"], bias="none"
)
model = get_peft_model(model, peft_config)

# Custom trainer applying class weights to cross-entropy loss
# Transformers >= 4.46 removed 'tokenizer' from Trainer; collator handles padding
class WeightedCELossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        logits = model(**inputs).logits
        loss = torch.nn.CrossEntropyLoss(weight=class_weights)(logits, labels)
        return (loss, {"logits": logits}) if return_outputs else loss

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average="binary")
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1": f1, "precision": precision, "recall": recall,
        "auc": roc_auc_score(p.label_ids, p.predictions[:, 1])
    }

# T4-optimized training arguments
args = TrainingArguments(
    output_dir=str(Path(BASE_DIR) / "models/lora"),
    learning_rate=2e-4, lr_scheduler_type="cosine",
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    per_device_eval_batch_size=4, num_train_epochs=2,
    weight_decay=0.01, max_grad_norm=0.5, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1", save_total_limit=2, logging_steps=100,
    report_to="none", dataloader_num_workers=2
)

trainer = WeightedCELossTrainer(
    model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["validation"],
    data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics
)

print("🚀 Starting LoRA fine-tuning...")
trainer.train()
trainer.save_model(str(Path(BASE_DIR) / "models/lora/best"))
print("✅ LoRA training complete. Best checkpoint saved.")

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Starting LoRA fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
1,0.011589,0.063942,0.984236,0.954733,0.987701,0.923894,0.997868
2,0.006804,0.054589,0.989172,0.969342,0.988051,0.951327,0.998018


✅ LoRA training complete. Best checkpoint saved.


# Cell 7: Export to ONNX & Inference Test

In [27]:
!pip install -q onnx onnxscript onnxruntime

import torch
from transformers import AutoTokenizer
from pathlib import Path

onnx_dir = Path(BASE_DIR) / "models/onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / "trace_esm2_lora.onnx"

# Merge LoRA adapters into base weights & move to CPU for stable ONNX tracing
merged_model = model.merge_and_unload()
merged_model.eval().cpu()

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
dummy_input = tokenizer("MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG", return_tensors="pt")

print("📦 Exporting to ONNX (CPU-optimized)...")
torch.onnx.export(
    merged_model,
    (dummy_input["input_ids"], dummy_input["attention_mask"]),
    str(onnx_path),
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}},
    opset_version=14,
    do_constant_folding=True
)
print(f"✅ ONNX model saved to {onnx_path}")

# Quick ONNX Runtime validation
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
onnx_inputs = {
    "input_ids": dummy_input["input_ids"].numpy(),
    "attention_mask": dummy_input["attention_mask"].numpy()
}
onnx_logits = session.run(None, onnx_inputs)[0]
print(f"🧪 ONNX inference test passed. Logits shape: {onnx_logits.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 10.0 MB/s eta 0:00:00
📦 Exporting to ONNX (CPU-optimized)...


/tmp/ipykernel_4445/656500485.py:19: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0426 17:41:05.311000 4445 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0426 17:41:06.529000 4445 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = 

[torch.onnx] Obtain model graph for `EsmForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...


/usr/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.esm.encoder.layer.11.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.11.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.1.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.1.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.10.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.10.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.12.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.12.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.20.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.20.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.13.attention.self.rotary_embeddings._cos_cached, self.esm.encoder.layer.13.attention.self.rotary_embeddings._sin_cached, self.esm.encoder.layer.7.attention.self.rotary_embeddings._cos_c

[torch.onnx] Obtain model graph for `EsmForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


...

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/exporter/_onnx_program.py:460: UserWarning: # The axis name: seq will not be used, since it shares the same shape constraints with another axis: seq.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/no_previous_version.h:24: adapt: Assertion `

✅ ONNX model saved to /content/drive/MyDrive/01-Research/TRACE/models/onnx/trace_esm2_lora.onnx
🧪 ONNX inference test passed. Logits shape: (1, 2)


# Cell 8: Output Summary & Local Handoff Instructions

In [36]:
import shutil, json

summary = {
    "workspace": BASE_DIR,
    "data_public": str(Path(BASE_DIR)/"data/public"),
    "data_generated": str(Path(BASE_DIR)/"data/generated"),
    "data_packaged": str(Path(BASE_DIR)/"data/packaged"),
    "lora_checkpoint": str(Path(BASE_DIR)/"models/lora/best"),
    "onnx_model": str(onnx_path),
    "test_carts": str(Path(BASE_DIR)/"data/generated/test_carts.json"),
    "motif_patterns": str(Path(BASE_DIR)/"data/public/motif_patterns.json")
}

with open(Path(BASE_DIR)/"outputs/trace_paths.json", "w") as f:
    json.dump(summary, f, indent=2)

print("📂 TRACE OUTPUT MAP:")
for k,v in summary.items():
    print(f"  {k}: {v}")
print("\n✅ All outputs saved to Google Drive. Ready for local API/Streamlit deployment.")

📂 TRACE OUTPUT MAP:
  workspace: /content/drive/MyDrive/01-Research/TRACE
  data_public: /content/drive/MyDrive/01-Research/TRACE/data/public
  data_generated: /content/drive/MyDrive/01-Research/TRACE/data/generated
  data_packaged: /content/drive/MyDrive/01-Research/TRACE/data/packaged
  lora_checkpoint: /content/drive/MyDrive/01-Research/TRACE/models/lora/best
  onnx_model: /content/drive/MyDrive/01-Research/TRACE/models/onnx/trace_esm2_lora.onnx
  test_carts: /content/drive/MyDrive/01-Research/TRACE/data/generated/test_carts.json
  motif_patterns: /content/drive/MyDrive/01-Research/TRACE/data/public/motif_patterns.json

✅ All outputs saved to Google Drive. Ready for local API/Streamlit deployment.


# Metric

In [ ]:
import torch
import numpy as np
import json
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
from datasets import load_from_disk
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, roc_auc_score,
    average_precision_score, precision_recall_curve
)

BASE_DIR = Path("/content/drive/MyDrive/01-Research/TRACE")
MODEL_DIR = BASE_DIR / "models/lora/best"
DATASET_DIR = BASE_DIR / "data/packaged"

# 1. Load model & tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
base_model = AutoModelForSequenceClassification.from_pretrained(
    "facebook/esm2_t33_650M_UR50D", num_labels=2, ignore_mismatched_sizes=True
)
model = PeftModel.from_pretrained(base_model, str(MODEL_DIR))
model.eval().to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# 2. Load validation dataset & batch loader
ds = load_from_disk(str(DATASET_DIR))
val_ds = ds["validation"]

# FIX: Tokenize the raw text dataset to generate input_ids and attention_mask
val_ds = val_ds.map(
    lambda b: tokenizer(b["sequence"], truncation=True, padding="max_length", max_length=512),
    batched=True
)
# Rename 'label' to 'labels' to match model expectations
val_ds = val_ds.rename_column("label", "labels")

# Set torch format
val_ds = val_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)

# 3. Batch inference
logits_list, labels_list = [], []
with torch.no_grad():
    for batch in loader:
        inputs = {k: v.to(model.device) for k, v in batch.items() if k != "labels"}
        logits_list.append(model(**inputs).logits.cpu().numpy())
        labels_list.append(batch["labels"].numpy())

logits = np.concatenate(logits_list)
labels = np.concatenate(labels_list)

# 4. Temperature scaling calibration
temp_path = MODEL_DIR / "temperature.npy"
temperature = float(np.load(temp_path)) if temp_path.exists() else 1.0
calibrated_logits = logits / temperature
probs = torch.softmax(torch.tensor(calibrated_logits), dim=1)[:, 1].numpy()

# 5. Operational threshold optimization
precisions, recalls, thresholds = precision_recall_curve(labels, probs)
valid = (recalls >= 0.95) & (precisions >= 0.98)
threshold = float(thresholds[valid][0]) if valid.any() else 0.5
preds = (probs >= threshold).astype(int)

# 6. Core metrics
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
metrics = {
    "accuracy": accuracy_score(labels, preds),
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc_score(labels, probs),
    "pr_auc": average_precision_score(labels, probs),
    "temperature": temperature,
    "operational_threshold": threshold,
    "n_samples": len(labels),
    "n_positives": int(labels.sum())
}

# 7. Expected Calibration Error
bin_edges = np.linspace(0, 1, 11)
ece = sum(
    ((probs >= bin_edges[i]) & (probs < bin_edges[i+1])).mean() *
    abs(probs[(probs >= bin_edges[i]) & (probs < bin_edges[i+1])].mean() - labels[(probs >= bin_edges[i]) & (probs < bin_edges[i+1])].mean())
    for i in range(10) if ((probs >= bin_edges[i]) & (probs < bin_edges[i+1])).any()
)
metrics["ece"] = ece

# 8. Output
print("📊 TRACE Evaluation Report")
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}" if isinstance(v, float) else f"  {k:25s}: {v}")

(MODEL_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"\n✅ Metrics saved to {MODEL_DIR / 'metrics.json'}")

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/6280 [00:00<?, ? examples/s]